# HProbe Tutorial — Discovering H-Neurons in Medical LLMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyincavusbi/hprobes/blob/main/hprobes_tutorial.ipynb)

**hprobes** discovers *H-Neurons* — a sparse set of FFN neurons whose activations predict (and causally influence) hallucination.

It uses the **CETT metric** (Contribution to rEsidual sTream norm of Token t):

$$\text{CETT}(j, t) = |z_{j,t}| \cdot \frac{\|W_{\text{down}}[:, j]\|_2}{\|h_t\|_2}$$

**This notebook covers:**
1. Installation & setup
2. Load model + dataset
3. `fit()` — discover H-Neurons
4. `score()` — AUROC vs random baseline
5. `causal_validate()` — causal intervention
6. `save()` / `load()` — persistence
7. `score_on()` — transfer experiments
8. Custom data & open-ended QA
9. CLI usage
10. Visualisations

**Runtime:** Set `Runtime → Change runtime type → T4 GPU`.

## 1. Installation

In [1]:
!pip install hprobes transformers accelerate datasets matplotlib -q

In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU:  Tesla T4
VRAM: 15.6 GB


## 2. HuggingFace Login

Gemma models are gated. Accept the license at https://huggingface.co/google/gemma-3-1b-it

Add your HF token to **Colab Secrets** (key icon in the left sidebar) with the name `HF_TOKEN`, then run the cell below. If this fails you can enter your token manually. 

In [3]:
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:
    import getpass

    token = getpass.getpass("HF Token: ")

from huggingface_hub import login

login(token=token)
print("Logged in OK")

Logged in OK


## 3. Load Model & Tokenizer

**Gemma-3-1B-IT** fits comfortably on a T4.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-3-1b-it"

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map=device,
)
model.eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Parameters: {n_params:.2f}B")
if device == "cuda":
    print(f"VRAM used:  {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading google/gemma-3-1b-it...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Parameters: 1.00B
VRAM used:  2.00 GB


## 4. Load Dataset

We use **MedQA** (USMLE-style questions)

In [5]:
from datasets import load_dataset

raw = load_dataset("openlifescienceai/medqa", split="test")

# Flatten the nested 'data' field into top-level keys
samples = []
for row in raw.select(range(200)):
    d = row["data"]
    samples.append(
        {
            "question": d["Question"],
            "options": d["Options"],  # {"A": "...", "B": "...", "C": "...", "D": "..."}
            "answer_idx": d["Correct Option"],  # "A" / "B" / "C" / "D"
        }
    )

print(f"Loaded {len(samples)} samples")
s = samples[0]
print(f"\nQuestion: {s['question'][:120]}...")
print(f"Options:  {s['options']}")
print(f"Answer:   {s['answer_idx']}")

Loaded 200 samples

Question: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending...
Options:  {'A': 'Disclose the error to the patient and put it in the operative report', 'B': 'Tell the attending that he cannot fail to disclose this mistake', 'C': 'Report the physician to the ethics committee', 'D': 'Refuse to dictate the operative report'}
Answer:   B


## 5. Discover H-Neurons with `fit()`

In contrastive mode, `fit()` runs **two batched forward passes** per batch:

1. **Prompt pass** (`forward_cett_batch`): tokenizes the prompt, forward pass, captures CETT at the **last prompt token** and reads logits to predict the answer letter
2. **Answer pass** (`forward_cett_at_token_batch`): appends the predicted letter token, forward pass, captures CETT at that **answer token position**

Each sample then contributes **two rows** to the feature matrix — one per pass — with **3-vs-1 labels**:
- Answer-token CETT → `1` (hallucinatory) if prediction is wrong, `0` if correct
- Last-prompt-token CETT → always `0` (negative baseline)

Finally, an **L1 logistic regression** is trained on the top-5000 highest-variance features to select the sparse set of H-Neurons (non-zero coefficients with positive weight).

In [7]:
from hprobes import HProbe

probe = HProbe(
    model=model,
    tokenizer=tokenizer,
    l1_C=0.5,  # inverse L1 strength — lower = sparser
    contrastive=True,  # contrastive 3-vs-1 labeling (recommended)
    validation_split=0.2,
    seed=42,
)

probe.fit(
    samples,
    question_key="question",
    options_key="options",
    answer_key="answer_idx",
)

[hprobes] Layers: 26  |  Features: 179,712  |  Contrastive: True


CETT extraction: 100%|██████████| 200/200 [01:38<00:00,  2.04it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


[hprobes] Valid: 200  |  Accuracy: 0.295
[hprobes] H-Neurons: 2481  |  Ratio: 13.805‰
[hprobes] Top layers: [(9, 236), (10, 223), (11, 208), (12, 186), (8, 169)]


In [8]:
print(f"Model accuracy:    {probe.accuracy_:.1%}")
print(f"H-Neurons found:   {probe.n_neurons_}  ({probe.neuron_ratio_:.3f}‰ of all features)")
print("\nTop layers:")
top = sorted(probe.layer_distribution_.items(), key=lambda x: -x[1])[:10]
for layer, count in top:
    print(f"  Layer {layer:3d}: {count:3d}  {'█' * (count // 2)}")

Model accuracy:    29.5%
H-Neurons found:   2481  (13.805‰ of all features)

Top layers:
  Layer   9: 236  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Layer  10: 223  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Layer  11: 208  ████████████████████████████████████████████████████████████████████████████████████████████████████████
  Layer  12: 186  █████████████████████████████████████████████████████████████████████████████████████████████
  Layer   8: 169  ████████████████████████████████████████████████████████████████████████████████████
  Layer  15: 160  ████████████████████████████████████████████████████████████████████████████████
  Layer   7: 147  █████████████████████████████████████████████████████████████████████████
  Layer  14: 130  █████████████████████████████████████████████████████████████████
  Layer  13: 116  █████████████

## 6. Score — AUROC vs Random Baseline

`score()` evaluates on the held-out validation split and computes a **random neuron baseline** — a probe trained on the same number of randomly selected neurons.

In [9]:
results = probe.score()

print(f"Probe AUROC:       {results['auroc']:.3f}")
print(f"Random baseline:   {results['random_baseline_auroc']:.3f}")
print(f"AUROC gap:         {results['auroc_gap']:+.3f}")
print(f"Balanced accuracy: {results['balanced_accuracy']:.3f}")
print(f"H-Neurons:         {results['n_h_neurons']}  ({results['neuron_ratio_permille']:.3f}‰)")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


[hprobes] AUROC: 0.854  |  Random: 0.869  |  Gap: -0.014
Probe AUROC:       0.854
Random baseline:   0.869
AUROC gap:         -0.014
Balanced accuracy: 0.849
H-Neurons:         2481  (13.805‰)


## 7. Causal Validation

`causal_validate()` scales H-Neuron activations by `alpha` and measures accuracy:
- `alpha = 0.0` — fully suppress (should **reduce** accuracy if causal)
- `alpha = 1.0` — baseline
- `alpha = 2.0` — amplify

In [10]:
cv = probe.causal_validate(alphas=[0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0])

baseline = cv[1.0]
print("Alpha  → Accuracy  (delta)")
for alpha, acc in sorted(cv.items()):
    delta = acc - baseline
    tag = " ← baseline" if alpha == 1.0 else f" ({delta:+.3f})"
    print(f"  {alpha:.2f}   {acc:.3f}{tag}")

Alpha  → Accuracy  (delta)
  0.00   0.050 (-0.250)
  0.25   0.300 (+0.000)
  0.50   0.200 (-0.100)
  0.75   0.225 (-0.075)
  1.00   0.300 ← baseline
  1.50   0.225 (-0.075)
  2.00   0.200 (-0.100)


In [ ]:
import matplotlib.pyplot as plt

alphas_plt = sorted(cv.keys())
accs_plt = [cv[a] for a in alphas_plt]
baseline = cv[1.0]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas_plt, accs_plt, "o-", color="steelblue", lw=2, ms=8)
ax.axhline(baseline, color="gray", ls="--", alpha=0.7, label=f"Baseline ({baseline:.3f})")
ax.axvline(1.0, color="gray", ls=":", alpha=0.5)
ax.fill_between(alphas_plt, baseline, accs_plt, alpha=0.15, color="steelblue")
ax.set_xlabel("Alpha (H-Neuron scaling factor)", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title(f"Causal Validation — {MODEL_ID.split('/')[-1]} on MedQA", fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Save & Load

`save()` writes:
- **`.json`** — human-readable results (H-Neurons, AUROC, causal validation)
- **`.pkl`** — serialised classifier for transfer experiments

In [12]:
import json

path = probe.save("results/gemma3_1b_medqa")
print(f"Saved to: {path}")

data = json.loads(path.read_text())
print("\nJSON keys:", list(data.keys()))
print("fit keys: ", list(data["fit"].keys()))
print(f"H-Neurons (first 3): {data['fit']['h_neurons'][:3]}")

Saved to: results/gemma3_1b_medqa.json

JSON keys: ['saved_at', 'fit', 'score', 'causal_validation']
fit keys:  ['n_h_neurons', 'neuron_ratio_permille', 'accuracy', 'layer_distribution', 'h_neurons']
H-Neurons (first 3): [[8, 1906], [0, 4161], [7, 1059]]


In [13]:
loaded_probe = HProbe.load("results/gemma3_1b_medqa", model=model, tokenizer=tokenizer)
print(f"Loaded probe: {loaded_probe.n_neurons_} H-Neurons")
print(f"First 5: {loaded_probe.h_neurons_[:5]}")

Loaded probe: 2481 H-Neurons
First 5: [(8, 1906), (0, 4161), (7, 1059), (4, 4597), (21, 5122)]


## 9. Transfer Experiments

`score_on()` tests H-Neurons discovered on MedQA on a **different dataset** (MMLU Medical).  
This answers: do H-Neurons encode general hallucination circuitry or dataset-specific patterns?

In [15]:
mmlu_raw = load_dataset("cais/mmlu", "medical_genetics", split="test")
mmlu_samples = [dict(row) for row in mmlu_raw]
print(f"MMLU Medical Genetics: {len(mmlu_samples)} samples")
print("Keys:", list(mmlu_samples[0].keys()))
print("Sample:", mmlu_samples[0])

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

medical_genetics/test-00000-of-00001.par(…):   0%|          | 0.00/16.4k [00:00<?, ?B/s]

medical_genetics/validation-00000-of-000(…):   0%|          | 0.00/5.63k [00:00<?, ?B/s]

medical_genetics/dev-00000-of-00001.parq(…):   0%|          | 0.00/3.77k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

MMLU Medical Genetics: 100 samples
Keys: ['question', 'subject', 'choices', 'answer']
Sample: {'question': 'In a Robertsonian translocation fusion occurs at the:', 'subject': 'medical_genetics', 'choices': ['telomeres.', 'centromeres.', 'histones.', 'ends of the long arms.'], 'answer': 1}


In [16]:
transfer = loaded_probe.score_on(
    mmlu_samples,
    question_key="question",
    options_key="choices",  # MMLU uses 'choices' list
    answer_key="answer",  # int index: 0=A, 1=B, ...
)

print("Transfer: MedQA H-Neurons → MMLU Medical Genetics")
print(f"  AUROC:           {transfer['auroc']:.3f}")
print(f"  Balanced acc:    {transfer['balanced_accuracy']:.3f}")
print(f"  Random baseline: {transfer['random_baseline_auroc']:.3f}")
print(f"  AUROC gap:       {transfer['auroc_gap']:+.3f}")

CETT extraction (transfer): 100%|██████████| 100/100 [00:26<00:00,  3.77it/s]

[hprobes transfer] AUROC: 0.501  |  Random: 0.559  |  Gap: -0.057
Transfer: MedQA H-Neurons → MMLU Medical Genetics
  AUROC:           0.501
  Balanced acc:    0.508
  Random baseline: 0.559
  AUROC gap:       -0.057


## 10. Custom Data

Pass any dataset with a `prompt_fn` — a callable that takes a sample dict and returns a formatted string.

In [17]:
def my_prompt_fn(sample):
    return (
        f"{sample['q']}\n"
        f"A. {sample['a']}\nB. {sample['b']}\nC. {sample['c']}\nD. {sample['d']}\n"
        "\nAnswer:"
    )


# custom_probe = HProbe(model, tokenizer, l1_C=0.5, batch_size=16)
# custom_probe.fit(
#     my_samples,
#     answer_key='correct',
#     prompt_fn=my_prompt_fn,
#     answer_cue='',           # already included in prompt_fn
# )
print("prompt_fn example:")
print(
    my_prompt_fn(
        {"q": "Which vitamin deficiency causes scurvy?", "a": "C", "b": "D", "c": "B12", "d": "A"}
    )
)

prompt_fn example:
Which vitamin deficiency causes scurvy?
A. C
B. D
C. B12
D. A

Answer:


## 11. Open-Ended QA — `fit_from_responses()`

For free-text tasks, capture CETT over the **answer token span** in a full Q+A sequence.

In [18]:
# Data format for fit_from_responses()
response_sample = {
    "question": "What is the mechanism of action of metformin?",
    "response": "Metformin inhibits mitochondrial complex I, reducing hepatic gluconeogenesis.",
    "answer_tokens": ["complex", "I"],  # factual span to capture CETT over
    "judge": "true",  # 'true' = correct, 'false' = hallucination
}

# response_probe = HProbe(model, tokenizer, l1_C=0.5)
# response_probe.fit_from_responses(
#     response_samples,
#     question_key='question',
#     response_key='response',
#     answer_tokens_key='answer_tokens',
#     label_key='judge',
#     aggregation='mean',   # or 'max'
# )
print("fit_from_responses() data format:")
for k, v in response_sample.items():
    print(f"  {k}: {v}")

fit_from_responses() data format:
  question: What is the mechanism of action of metformin?
  response: Metformin inhibits mitochondrial complex I, reducing hepatic gluconeogenesis.
  answer_tokens: ['complex', 'I']
  judge: true


## 12. CLI Usage

The same functionality is available from the command line for large-scale runs.

In [19]:
!hprobes --help

usage: hprobes [-h] [--version] {run,responses,transfer} ...

Hallucination neuron probe — discover and causally validate H-Neurons

positional arguments:
  {run,responses,transfer}
    run                 Fit, score, and causal-validate on an MCQ dataset
    responses           Fit from pre-generated responses (open-ended / free-
                        text)
    transfer            Score a saved probe on a different model (transfer
                        experiment)

options:
  -h, --help            show this help message and exit
  --version             show program's version number and exit


In [20]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
with open("data/medqa_200.jsonl", "w") as f:
    for s in samples:
        f.write(json.dumps(s) + "\n")
print("Saved data/medqa_200.jsonl")

Saved data/medqa_200.jsonl


In [22]:
%%bash
hprobes run \
  --model google/gemma-3-1b-it \
  --data data/medqa_200.jsonl \
  --format medqa \
  --l1-c 0.5 \
  --output results/cli_demo


hprobes v0.3.0  |  model: google/gemma-3-1b-it  |  samples: 100
────────────────────────────────────────────────────────────────────
  Dataset:     medqa_200.jsonl  (100 samples, format=medqa)
  Keys:        options_key='options'  answer_key='answer_idx'
  Contrastive: True
  Loading google/gemma-3-1b-it... done
  Fitting (l1_C=0.5)...[hprobes] Layers: 26  |  Features: 179,712  |  Contrastive: True
[hprobes] Valid: 100  |  Accuracy: 0.320
[hprobes] H-Neurons: 2463  |  Ratio: 13.705‰
[hprobes] Top layers: [(9, 221), (10, 210), (11, 196), (12, 186), (15, 156)]
 done
  H-Neurons:   2463  (13.705‰ of all features)
  Accuracy:    0.320
  Layers:      {0: 52, 1: 55, 2: 45, 3: 60, 4: 71, 5: 99, 6: 104, 7: 130, 8: 153, 9: 221, 10: 210, 11: 196, 12: 186, 13: 119, 14: 146, 15: 156, 16: 70, 17: 100, 18: 46, 19: 38, 20: 41, 21: 36, 22: 16, 23: 26, 24: 21, 25: 66}

  Scoring...
[hprobes] AUROC: 0.929  |  Random: 0.926  |  Gap: +0.003
  AUROC:           0.929
  Random baseline: 0.926
  AUROC gap:  

CETT extraction: 100%|██████████| 100/100 [00:51<00:00,  1.95it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


In [24]:
# Transfer via CLI:
# !hprobes transfer \
#   --probe results/cli_demo \
#   --model google/gemma-3-1b-it \
#   --data data/mmlu_medical.jsonl \
#   --format mmlu \
#   --output results/cli_transfer

## 13. Visualising Results

In [ ]:
import numpy as np
from hprobes.cett import available_layers

n_layers = len(available_layers(model))
counts = np.zeros(n_layers, dtype=int)
for layer_idx, _ in probe.h_neurons_:
    counts[layer_idx] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.bar(range(n_layers), counts, color=["steelblue" if c > 0 else "#ddd" for c in counts])
ax.set_xlabel("Layer", fontsize=12)
ax.set_ylabel("H-Neurons", fontsize=12)
ax.set_title(
    f"H-Neuron Distribution by Layer\n({probe.n_neurons_} total, {probe.neuron_ratio_:.3f}‰)",
    fontsize=12,
)
ax.grid(axis="y", alpha=0.3)

ax = axes[1]
sr = probe.score_results_
bars = ax.bar(
    ["H-Neurons", "Random"],
    [sr["auroc"], sr["random_baseline_auroc"]],
    color=["steelblue", "#aaa"],
    width=0.5,
)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("AUROC", fontsize=12)
ax.set_title("Probe vs Random Baseline", fontsize=12)
for bar, val in zip(bars, [sr["auroc"], sr["random_baseline_auroc"]]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.005,
        f"{val:.3f}",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
    )
ax.grid(axis="y", alpha=0.3)

plt.suptitle(f"{MODEL_ID.split('/')[-1]} — MedQA (200 samples)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

| Step | Method | What it does |
|------|--------|--------------|
| Discover | `probe.fit(samples)` | Find sparse H-Neurons via CETT + L1 probe |
| Evaluate | `probe.score()` | AUROC on held-out split vs random baseline |
| Validate | `probe.causal_validate()` | Do suppressing H-Neurons change behaviour? |
| Save | `probe.save(path)` | Write JSON results + PKL classifier |
| Load | `HProbe.load(path, model, tok)` | Reload saved probe |
| Transfer | `probe.score_on(new_samples)` | Test H-Neurons on a different dataset |
| Custom | `fit(prompt_fn=my_fn)` | Any MCQ format |
| Open QA | `fit_from_responses(samples)` | Free-text generation tasks |
| CLI | `hprobes run --model ... --data ...` | Batch experiments |

### Key parameters

| Parameter | Default | Effect |
|-----------|---------|--------|
| `l1_C` | 0.5 | Sparsity — lower = fewer H-Neurons |
| `contrastive` | True | 3-vs-1 labeling at answer token (recommended) |
| `batch_size` | 1 | GPU batch size — set 16–32 for speed |
| `layer_stride` | 1 | 2 = every other layer (2× faster) |

### Links
- PyPI: https://pypi.org/project/hprobes/
- GitHub: https://github.com/huseyincavusbi/hprobes
- CETT paper: arXiv:2512.01797